# K Nearest Neighbors

## Algorithm

KNN is a classification/regression algorithm for supervised learning. 

For classification problem, it assigns a class label to a data point based on the class labels of its k nearest neighbors in the training set.
- choose a value of k, the number of the nearest neighbors to consider
- for each data point in the test data, compute its distance to all data points in the training set
- select k nearest neighbors based on their distance for each data point in the test data set
- predict based on the labels of k nearest neighbors
    - For classification problem, assign the class label that appears most frequently among the k nearest neighbors to the test point. 
    - For regression, just average the k nearest neighbors
 


## Inputs
- $X_{train} \rightarrow [N, D]$
- $Y_{train} \rightarrow [N]$
- $X_{test} \rightarrow [M, D]$
- $k$

## Numpy Implementation

In [28]:
import numpy as np

In [ ]:
class KNN:
    def __init__(self, k=5, distance = 'euclidean'):
        self.k = k
        self.distance = distance
    
    def fit(self, X, y):
        self.X_train = X # (N, D)
        self.y_train = y # (N,)
    
    def predict(self, X):
        # compute distance between test and train samples
        # (M, N)
        if self.distance == 'euclidean':
            dists = np.sqrt(np.sum((X[:, np.newaxis, :] - self.X_train[np.newaxis, :, :]) ** 2, axis=2))
        elif self.distance == 'manhattan':
            dists = np.sum(np.abs(X[:, np.newaxis, :] - self.X_train[np.newaxis, :, :]), axis=2)
        else:
            raise ValueError("Unsupported distance metric")
        
        # find the k nearest neighbors for each test sample
        # (M, k)
        indices = np.argpartition(dists, kth=self.k, axis=1)[:, :self.k]
        
        # get the labels of the k nearest neighbors
        # (M, k) 
        nearest_labels = self.y_train[indices] 

        # majority vote
        y_pred = self._vote(nearest_labels)
        return y_pred

    def _vote(self, nearest_labels):
        # (M, k) -> (M,)
        M = nearest_labels.shape[0]
        y_pred = np.zeros(M, dtype=int)

        for i in range(M):
            # count the occurrences of each class in the k nearest neighbors
            counts = np.bincount(nearest_labels[i])
            # assign the class with the highest count as the predicted label
            y_pred[i] = np.argmax(counts)

        return y_pred

In [30]:
N, D, M, k = 500, 10, 100, 5
X_train = np.random.rand(N, D)
y_train = np.random.randint(0, 3, size=N)
X_test = np.random.rand(M, D)

knn = KNN(k=k, distance='euclidean')
knn.fit(X_train, y_train)
y_pred = knn.predict(X_test)
print(y_pred)

[2 0 0 0 1 0 0 0 2 1 2 0 1 0 1 0 0 2 1 1 2 1 0 0 0 0 1 0 0 1 2 0 2 0 0 1 2
 0 0 0 0 2 0 0 0 2 1 0 0 2 1 0 0 0 1 2 2 1 1 1 1 1 1 0 0 0 0 0 0 1 2 0 2 2
 2 2 2 0 0 0 0 2 1 2 1 0 1 1 0 0 1 0 0 1 0 0 1 2 1 2]


## Further Improvements

- weighted KNN, where add weigths to the neareast neighbor. The weights can be a function of distance as:
```python
    weights = 1./(dists + 1e-08)
```

- approximate KNN
    - KD-tree
    - Ball-tree
    - FAISS (important in real system as KNN is slow in inference time)

- scaling issues
    - use batch if M, N is large
    - use GPU for matrix multiplication